# Lab 06: Environments (Proxy Verification)
Running the AI's config through the actual SCALE-Sim engine.


In [ ]:
import json
import os

with open('.arch2_state.json', 'r') as f: state = json.load(f)
config = state.get('ai_config', {"ArrayHeight": 8, "ArrayWidth": 8})

scale_cfg = f"""[general]
run_name = ai_run
[architecture_presets]
ArrayHeight: {config.get('ArrayHeight', 8)}
ArrayWidth: {config.get('ArrayWidth', 8)}
IfmapSramSzkB: 256
FilterSramSzkB: 256
OfmapSramSzkB: 128
Dataflow: os
Bandwidth: 10
MemoryBanks: 1
"""
with open('scale_sim.cfg', 'w') as f: f.write(scale_cfg)

with open('topo.csv', 'w') as f:
    f.write("Layer name,IFMAP Height,IFMAP Width,Filter Height,Filter Width,Channels,Num Filter,Strides\n")
    f.write("conv1,32,32,3,3,64,64,1\n")

print("SCALE-sim config generated. Running simulation...")



In [ ]:
import subprocess
import pandas as pd
try:
    # Attempt real SCALE-Sim run
    from scalesim.scale_sim import scalesim
    sim = scalesim(save_disk_space=True, verbose=False, config='scale_sim.cfg', topology='topo.csv')
    sim.run_scale(top_path='.')
    df = pd.read_csv('ai_run_COMPUTE_REPORT.csv')
    total_cycles = int(df['Total Cycles'].sum())
except Exception as e:
    print("Real SCALE-Sim run failed or not installed. Mocking cycles based on Area for proxy.")
    # Fallback proxy equation: larger arrays = faster
    total_cycles = 10000000 // (config.get('ArrayHeight', 8) * config.get('ArrayWidth', 8))

print(f"Proxy Result: {total_cycles} cycles.")
state['proxy_cycles'] = total_cycles
with open('.arch2_state.json', 'w') as f: json.dump(state, f)
